In [ ]:
from utils.paths import DESCRIPTORS_DIR,DATABASE_DIR

In [ ]:
import numpy as np
import pandas as pd
from preprocessing.DescriptorEngineer import AlloyDescriptorCalculator,ElementPropertyLoader
from preprocessing.database_manager import DatabaseManager
from preprocessing.DataPreprocessor import DataPreprocessor
from models_scripts.AdaptiveTransferKernel import AdaptiveTransferKernel

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel,ConstantKernel,Matern
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
db = DatabaseManager(DATABASE_DIR/"Ternary_round1.db")

df_optical = db.table_dataframe('Optical_properties')
df_comp = db.table_dataframe('compositions')
db.close()

# --- Merge and filter wavelength ---
df_merged = df_optical.merge(df_comp, on="ID", how="inner")
# wavelength filter
tern_1550 = df_merged[np.isclose(df_merged["wavelength_nm"].astype(float), 1552.0)]

Connected to database: /Users/linarojas/Desktop/Research/Papers/Combinatorial_Ternary/Hybrid-Experimental-Data-Driven-Workflow/data/databases/Ternary_round1.db
Database connection closed


In [16]:
db = DatabaseManager(DATABASE_DIR/"Calculate_EMA.db")
df_comp_all = db.table_dataframe('compositions')
db.close()

Connected to database: /Users/linarojas/Desktop/Research/Papers/Combinatorial_Ternary/Hybrid-Experimental-Data-Driven-Workflow/data/databases/Calculate_EMA.db
Database connection closed


Intrinsic Parameters
1. Average Atomic Radius
2. Atomic Radius mismatch
3. Electronegativity difference
4. Mixing entropy
5. Average valence electron concentration

In [10]:
atomic_radius = pd.read_pickle(DESCRIPTORS_DIR/"Atomic_radius.pkl")
electronegativity = pd.read_pickle(DESCRIPTORS_DIR/"electronegativity.pkl")
valence_electrons = pd.read_pickle(DESCRIPTORS_DIR/"Valence_electrons.pkl")

atomic_radius = atomic_radius.set_index("symbol")
electronegativity = electronegativity.set_index("Symbol")
valence_electrons = valence_electrons.set_index("symbol")

atomic_radius = atomic_radius.rename(columns={"Metallic": "atomic_radius"})
valence_electrons = valence_electrons.rename(columns={"valence": "valence_electrons"})

elem_props = (
    atomic_radius
    .join(electronegativity)
    .join(valence_electrons)
)

In [19]:
calc = AlloyDescriptorCalculator(elem_props=elem_props)
df_ternary_1f_properties = calc.add_all_descriptors(tern_1550[['Cu','Ni','Al']])
comp_all_properties = calc.add_all_descriptors(df_comp_all[['Cu','Ni','Al']])

Experimental data GP - Random splitting

In [27]:
DP = DataPreprocessor()
X = df_ternary_1f_properties
y = tern_1550['e2']

X_train_split, X_test_split, y_train,y_test = DP.split_training(X,y)

X_train = X_train_split.drop(columns=['r','r_ave','S'])
X_test = X_test_split.drop(columns=['r','r_ave','S'])

In [25]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C

X_train = np.asarray(X_train, dtype=float)
y_train = np.asarray(y_train, dtype=float).ravel()

kernel = (
    C(1.0, (1e-3, 1e3)) *
    Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e2), nu=2.5)
    + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-8, 1e1))
)

gpr_nt = GaussianProcessRegressor(
    kernel=kernel,
    alpha=0.0,   
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

gpr_nt.fit(X_train, y_train)
print("Learned kernel:", gpr_nt.kernel_)

Learned kernel: 1.06**2 * Matern(length_scale=0.675, nu=2.5) + WhiteKernel(noise_level=0.248)


In [26]:
X_test = np.asarray(X_test, dtype=float)
y_pred, y_std_nt = gpr_nt.predict(X_test, return_std=True)

mae  = mean_absolute_error(y_pred, y_test)
rmse = np.sqrt(mean_squared_error(y_pred, y_test))
r2   = r2_score(y_pred, y_test)

print(f"MAE : {mae:.4g}")
print(f"RMSE: {rmse:.4g}")
print(f"R^2 : {r2:.4g}")

MAE : 7.241
RMSE: 7.832
R^2 : 0.7045


In [ ]:
x_all_space = np.asarray(comp_all_properties[['Cu','Ni','Al','VEC','del_r','del_EN']], dtype=float)
y_pred_exp, y_std_nt_exp = gpr_nt.predict(x_all_space, return_std=True)


Experimental data GP - Proportional splitting

In [28]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

Adaptive kernel approach

In [ ]:
# Source and target variables for Adaptive transfer kernel

def data_transfer_gp (X_source,X_target, y_source, y_target):
    # Convert to numpy
    Xs = np.asarray(X_source, dtype=float)
    Xt = np.asarray(X_target, dtype=float)
    
    # Add domain indicator ( 0 = source, 1 = target)
    Xs_aug = np.c_[Xs, np.zeros((len(Xs),1))]
    Xt_aug = np.c_[Xt, np.ones((len(Xt),1))]
    
    #combine datasets
    X_train = np.vstack([Xs_aug, Xt_aug])
    y_train = np.concatenate([y_source, y_target])
    
    return X_train,y_train

In [80]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor

kernel = AdaptiveTransferKernel(
    kernel=1.0 * Matern(length_scale=1.0, nu=2.5),
    lamb=2.0,
    lamb_bounds=(1.0, 3.0),
    different_noises=False,
)

gpr_pe = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-5,            # jitter for numerical stability
    normalize_y=True,      # y standardization inside sklearn
    n_restarts_optimizer=3 # increase later (5-10) if slow/unstable
)